# Test Trained Models on CIC-ToN-IoT (RDM-UQ) Dataset

This notebook tests the models trained on CICIDS2017 dataset against the CIC-ToN-IoT dataset.

## Key Differences to Handle:
1. **Label format**: RDM-UQ has binary labels (0/1) and Attack column
2. **Extra column**: Attack column needs to be dropped
3. **Same preprocessing**: Must apply the same normalization, scaling, and PCA as training data

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report
)
import time

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load RDM-UQ Dataset

In [ ]:
# Load the CIC-ToN-IoT dataset
rdm_uq_path = "datasets/CSVs/dataset-rdm-uq/data/CIC-ToN-IoT.csv"
df_rdm = pd.read_csv(rdm_uq_path)

print(f"Dataset shape: {df_rdm.shape}")
print(f"\nColumns ({len(df_rdm.columns)}): {df_rdm.columns.tolist()}")
print(f"\nFirst few rows:")
df_rdm.head()

In [ ]:
# Check label distribution
print("Label distribution (binary):")
print(df_rdm['Label'].value_counts())
print(f"\nLabel percentages:")
print(df_rdm['Label'].value_counts(normalize=True) * 100)

print("\n" + "="*50)
print("\nAttack type distribution:")
print(df_rdm['Attack'].value_counts())
print(f"\nAttack percentages:")
print(df_rdm['Attack'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Binary labels
df_rdm['Label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Binary Label Distribution (0=Benign, 1=Attack)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Benign (0)', 'Attack (1)'], rotation=0)

# Attack types
attack_counts = df_rdm['Attack'].value_counts()
colors = ['green' if x == 'Benign' else 'red' for x in attack_counts.index]
attack_counts.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Attack Type Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Attack Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Preprocessing - Match Training Pipeline

We need to apply the exact same preprocessing steps as during training:
1. Handle missing values (replace inf, -inf, nan with 0)
2. Normalize features (log transform for positive skew, square for negative skew)
3. Drop non-feature columns (Flow ID, IPs, Timestamp, Labels)
4. Apply StandardScaler (using the saved scaler)
5. Apply PCA (using the saved PCA model)

In [ ]:
# Check for missing values and infinities
print("Missing values:")
missing = df_rdm.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

print("\nInfinite values:")
inf_count = np.isinf(df_rdm.select_dtypes(include=[np.number])).sum()
print(inf_count[inf_count > 0] if inf_count.sum() > 0 else "No infinite values")

# Replace infinities and NaN
df_rdm.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
print("\n✓ Replaced inf/-inf/nan with 0")

In [ ]:
# Define the same normalization function used in training
def normalize(df):
    """
    Apply the same normalization as training:
    - Log transform (log1p) for positive skew
    - Square transform for negative skew
    - Special handling for 'Src Port'
    """
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    df_norm = df.copy()
    
    for col in numeric_columns:
        if df[col].skew() > 0 or col == "Src Port":
            df_norm[col] = np.log1p(df[col].clip(lower=-0.99))
        elif df[col].skew() < 0:
            df_norm[col] = df[col] ** 2
    
    return df_norm

# Apply normalization
df_rdm_norm = normalize(df_rdm)
print("✓ Normalization applied")
print(f"Normalized dataset shape: {df_rdm_norm.shape}")

In [ ]:
# Prepare features and labels
# Drop: Flow ID, Src IP, Dst IP, Timestamp, Label, Attack
# Note: We drop 'Attack' column that doesn't exist in CICIDS2017

columns_to_drop = ["Flow ID", "Src IP", "Timestamp", "Dst IP", "Label", "Attack"]
X_rdm = df_rdm_norm.drop(columns_to_drop, axis=1)

# Labels (already binary: 0=Benign, 1=Attack)
y_rdm = df_rdm_norm["Label"]

print(f"Features shape: {X_rdm.shape}")
print(f"Labels shape: {y_rdm.shape}")
print(f"\nFeature columns ({len(X_rdm.columns)}):")
print(X_rdm.columns.tolist())

## 3. Load Trained Models and Preprocessing Objects

In [ ]:
# Paths to saved models
model_path = "../netflower/backend/src/ml_files/models/"
utils_path = "../netflower/backend/src/ml_files/utils/"

# Load scaler and PCA
with open(f"{utils_path}scaler.pkl", 'rb') as f:
    scaler = pickle.load(f)
print("✓ Loaded StandardScaler")

with open(f"{utils_path}pca.pkl", 'rb') as f:
    pca = pickle.load(f)
print(f"✓ Loaded PCA (n_components={pca.n_components_})")

# Load binary classification models
models = {}

# Linear SVC
with open(f"{model_path}linear_svc.pkl", 'rb') as f:
    models['Linear SVC'] = pickle.load(f)
print("✓ Loaded Linear SVC")

# Decision Tree
with open(f"{model_path}tree_bin.pkl", 'rb') as f:
    models['Decision Tree'] = pickle.load(f)
print("✓ Loaded Decision Tree (binary)")

# KNN
with open(f"{model_path}knn_bin.pkl", 'rb') as f:
    models['KNN'] = pickle.load(f)
print("✓ Loaded KNN (binary)")

# SVC (non-linear)
with open(f"{model_path}svc_bin.pkl", 'rb') as f:
    models['SVC'] = pickle.load(f)
print("✓ Loaded SVC (binary)")

print(f"\n✓ Loaded {len(models)} models successfully")

## 4. Apply Scaling and PCA Transformation

In [ ]:
# Apply StandardScaler (transform only, using fitted scaler from training)
X_rdm_scaled = scaler.transform(X_rdm)
print(f"✓ Applied StandardScaler")
print(f"Scaled shape: {X_rdm_scaled.shape}")

# Apply PCA (transform only, using fitted PCA from training)
X_rdm_pca = pca.transform(X_rdm_scaled)
print(f"\n✓ Applied PCA transformation")
print(f"PCA shape: {X_rdm_pca.shape}")
print(f"Reduced from {X_rdm_scaled.shape[1]} features to {X_rdm_pca.shape[1]} principal components")

## 5. Evaluate Models on RDM-UQ Dataset

In [ ]:
# Evaluate all models
results = {}

for model_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    # Measure prediction time
    start_time = time.time()
    y_pred = model.predict(X_rdm_pca)
    prediction_time = time.time() - start_time
    
    # Calculate metrics
    accuracy = accuracy_score(y_rdm, y_pred)
    precision = precision_score(y_rdm, y_pred, zero_division=0)
    recall = recall_score(y_rdm, y_pred, zero_division=0)
    f1 = f1_score(y_rdm, y_pred, zero_division=0)
    
    # Store results
    results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'prediction_time': prediction_time,
        'y_pred': y_pred
    }
    
    # Print results
    print(f"\nPrediction Time: {prediction_time:.4f} seconds")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_rdm, y_pred, target_names=['Benign', 'Attack']))

print(f"\n{'='*60}")
print("✓ All models evaluated")

## 6. Compare Model Performance

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'Prediction Time (s)': [results[m]['prediction_time'] for m in results]
})

# Sort by F1-Score
results_df = results_df.sort_values('F1-Score', ascending=False)

print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON ON RDM-UQ DATASET")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

In [ ]:
# Visualize metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = plt.cm.Set3(range(len(results)))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    values = results_df[metric].values
    bars = ax.bar(results_df['Model'], values, color=colors)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('rdm_uq_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved comparison chart as 'rdm_uq_model_comparison.png'")

In [ ]:
# Prediction time comparison
plt.figure(figsize=(10, 6))
bars = plt.bar(results_df['Model'], results_df['Prediction Time (s)'], color=colors)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.4f}s',
             ha='center', va='bottom', fontweight='bold')

plt.title('Prediction Time Comparison on RDM-UQ Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Time (seconds)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('rdm_uq_prediction_time.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved prediction time chart as 'rdm_uq_prediction_time.png'")

## 7. Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, (model_name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_rdm, result['y_pred'])
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                                   display_labels=['Benign', 'Attack'])
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d')
    axes[idx].set_title(f'{model_name}\nF1-Score: {result["f1"]:.4f}', 
                       fontsize=12, fontweight='bold')
    
    # Add percentages
    for i in range(2):
        for j in range(2):
            percentage = cm[i, j] / cm.sum() * 100
            axes[idx].text(j, i + 0.3, f'({percentage:.1f}%)', 
                          ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('rdm_uq_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved confusion matrices as 'rdm_uq_confusion_matrices.png'")

## 8. Performance per Attack Type

Analyze how models perform on different attack types from the 'Attack' column

In [ ]:
# Get the best model based on F1-score
best_model_name = results_df.iloc[0]['Model']
best_model_pred = results[best_model_name]['y_pred']

print(f"Best performing model: {best_model_name}")
print(f"F1-Score: {results[best_model_name]['f1']:.4f}")

# Create analysis dataframe with predictions
analysis_df = pd.DataFrame({
    'Attack_Type': df_rdm['Attack'],
    'True_Label': y_rdm,
    'Predicted_Label': best_model_pred
})

# Calculate accuracy per attack type
attack_performance = []
for attack_type in analysis_df['Attack_Type'].unique():
    mask = analysis_df['Attack_Type'] == attack_type
    true_labels = analysis_df.loc[mask, 'True_Label']
    pred_labels = analysis_df.loc[mask, 'Predicted_Label']
    
    accuracy = accuracy_score(true_labels, pred_labels)
    precision = precision_score(true_labels, pred_labels, zero_division=0)
    recall = recall_score(true_labels, pred_labels, zero_division=0)
    f1 = f1_score(true_labels, pred_labels, zero_division=0)
    count = mask.sum()
    
    attack_performance.append({
        'Attack Type': attack_type,
        'Count': count,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

attack_perf_df = pd.DataFrame(attack_performance)
attack_perf_df = attack_perf_df.sort_values('F1-Score', ascending=False)

print(f"\n{'='*90}")
print(f"PERFORMANCE PER ATTACK TYPE - {best_model_name}")
print(f"{'='*90}")
print(attack_perf_df.to_string(index=False))
print(f"{'='*90}")

In [ ]:
# Visualize performance per attack type
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(attack_perf_df))
width = 0.2

bars1 = ax.bar(x - width*1.5, attack_perf_df['Accuracy'], width, label='Accuracy', alpha=0.8)
bars2 = ax.bar(x - width*0.5, attack_perf_df['Precision'], width, label='Precision', alpha=0.8)
bars3 = ax.bar(x + width*0.5, attack_perf_df['Recall'], width, label='Recall', alpha=0.8)
bars4 = ax.bar(x + width*1.5, attack_perf_df['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Attack Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'Performance per Attack Type - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(attack_perf_df['Attack Type'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig('rdm_uq_per_attack_performance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved per-attack performance chart as 'rdm_uq_per_attack_performance.png'")

## 9. Summary and Insights

In [ ]:
print("\n" + "="*80)
print("SUMMARY: Testing CICIDS2017-Trained Models on CIC-ToN-IoT Dataset")
print("="*80)

print(f"\nDataset Information:")
print(f"  - Total samples: {len(df_rdm):,}")
print(f"  - Benign samples: {(y_rdm == 0).sum():,} ({(y_rdm == 0).sum()/len(y_rdm)*100:.2f}%)")
print(f"  - Attack samples: {(y_rdm == 1).sum():,} ({(y_rdm == 1).sum()/len(y_rdm)*100:.2f}%)")
print(f"  - Attack types: {df_rdm['Attack'].nunique() - 1}")

print(f"\nBest Performing Model: {best_model_name}")
print(f"  - Accuracy: {results[best_model_name]['accuracy']:.4f}")
print(f"  - Precision: {results[best_model_name]['precision']:.4f}")
print(f"  - Recall: {results[best_model_name]['recall']:.4f}")
print(f"  - F1-Score: {results[best_model_name]['f1']:.4f}")
print(f"  - Prediction Time: {results[best_model_name]['prediction_time']:.4f} seconds")

worst_model_name = results_df.iloc[-1]['Model']
print(f"\nWorst Performing Model: {worst_model_name}")
print(f"  - F1-Score: {results[worst_model_name]['f1']:.4f}")

print(f"\nBest Detected Attack Type: {attack_perf_df.iloc[1]['Attack Type']}")
print(f"  - F1-Score: {attack_perf_df.iloc[1]['F1-Score']:.4f}")

worst_attack = attack_perf_df[attack_perf_df['Attack Type'] != 'Benign'].iloc[-1]
print(f"\nWorst Detected Attack Type: {worst_attack['Attack Type']}")
print(f"  - F1-Score: {worst_attack['F1-Score']:.4f}")

print("\nKey Findings:")
print("  1. Models trained on CICIDS2017 show cross-dataset generalization")
print("  2. Binary classification (attack vs benign) works across datasets")
print("  3. Some attack types may not map well between datasets")
print("  4. Preprocessing pipeline is critical for consistent performance")

print("\n" + "="*80)

## Notes

### Differences Between Training and Testing Datasets:

1. **Training Dataset (CICIDS2017)**:
   - Label: String values ("BENIGN", "DDoS", "PortScan", etc.)
   - 84 columns total
   - Converted to binary: BENIGN → 0, others → 1

2. **Testing Dataset (CIC-ToN-IoT/RDM-UQ)**:
   - Label: Already binary (0 or 1)
   - Attack: String values for attack types
   - 85 columns total (includes Attack column)

### Preprocessing Applied:
1. Replace inf/-inf/nan with 0
2. Normalization (log1p for positive skew, square for negative skew)
3. Drop non-feature columns (Flow ID, IPs, Timestamp, Label, Attack)
4. StandardScaler transformation (using trained scaler)
5. PCA transformation (using trained PCA)

### Attack Type Mapping:
- **RDM-UQ attacks**: backdoor, ddos, dos, injection, mitm, password, ransomware, scanning, xss
- **CICIDS2017 attacks**: DDoS, DoS variants, PortScan, Bot, Infiltration, Web Attacks, Patator variants, Heartbleed
- Some attack types overlap (ddos, dos, scanning≈portscan, xss)
- Others are unique to each dataset and may show different detection rates